**Zomato Restaurant Review Analysis using Machine Learning**

**Problem Statement**

The goal of this project is to build a machine learning model that analyzes restaurant data and customer reviews to predict the overall sentiment and performance of restaurants.

Using Zomato datasets containing restaurant metadata and user reviews, the model aims to:
Extract meaningful insights from textual reviews

Classify customer sentiment as positive or negative

Help improve restaurant recommendations and user decision-making

**Key Insights**

**Majority Positive Sentiment**:Most customer reviews are positive, showing that overall restaurant experiences are satisfactory.

**Service Impacts Negative Reviews**:Negative feedback is mainly due to slow service ,poor staff behavior, delivery delays.


**Food Quality Drives Ratings**:Restaurants with consistent food quality receive higher ratings and better reviews.

**Price ≠ Satisfaction**:Higher cost does not always mean better ratings — affordable restaurants can also perform very well.

**Text Data is Highly Valuable**:Customer reviews (text data) play a major role in predicting sentiment and understanding customer opinions.


**Summary**

This project focuses on analyzing Zomato restaurant data and customer reviews using Machine Learning and Natural Language Processing (NLP).

The objective is to understand customer opinions and predict sentiment (positive or negative) based on review text. The process involves data cleaning, text preprocessing, feature extraction using TF-IDF, and building classification models such as Random Forest and Logistic Regression.

The model learns patterns from customer reviews and accurately classifies sentiment, helping identify restaurant performance and customer satisfaction levels.

Overall, this project demonstrates how unstructured text data can be transformed into meaningful insights to improve restaurant recommendations and business decisions.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

**Data Handling Libraries**: pandas → used to work with datasets (CSV files, tables)

numpy → used for numerical operations

**Visualization Libraries**:Used to create graphs and chartsand helps in understanding data visually.

**Text Processing Libraries**:

re → removes unwanted characters (like symbols, numbers)

nltk → Natural Language Processing library

stopwords → removes common words like “is”, “the”, “and”

**Machine Learning Libraries**: Splits data into:

Training data (to learn)

Testing data (to check performance)

**Load Dataset**

In [2]:
# Load datasets
meta = pd.read_csv('/content/Zomato Restaurant names and Metadata.csv')
reviews = pd.read_csv('/content/Zomato Restaurant reviews.csv')

# Display first 5 rows
meta.head()
reviews.head()

,Restaurant,Reviewer,Review,Rating,Metadata,Time,Pictures
0,Beyond Flavours,Rusha Chakraborty,"The ambience was good, food was quite good . h...",5,"1 Review , 2 Followers",5/25/2019 15:54,0
1,Beyond Flavours,Anusha Tirumalaneedi,Ambience is too good for a pleasant evening. S...,5,"3 Reviews , 2 Followers",5/25/2019 14:20,0
2,Beyond Flavours,Ashok Shekhawat,A must try.. great food great ambience. Thnx f...,5,"2 Reviews , 3 Followers",5/24/2019 22:54,0
3,Beyond Flavours,Swapnil Sarkar,Soumen das and Arun was a great guy. Only beca...,5,"1 Review , 1 Follower",5/24/2019 22:11,0
4,Beyond Flavours,Dileep,Food is good.we ordered Kodi drumsticks and ba...,5,"3 Reviews , 2 Followers",5/24/2019 21:37,0


We are loading two datasets, meta dataset which contains details like name,
location,cost,cuisine and reviews dataset which contains customer reviews, ratings and review text.


**Data Understanding & Checking Missing Values**

In [3]:
# Check basic info
meta.info()
reviews.info()

# Check missing values
meta.isnull().sum()
reviews.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Name         105 non-null    object
 1   Links        105 non-null    object
 2   Cost         105 non-null    object
 3   Collections  51 non-null     object
 4   Cuisines     105 non-null    object
 5   Timings      104 non-null    object
dtypes: object(6)
memory usage: 5.1+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Restaurant  10000 non-null  object
 1   Reviewer    9962 non-null   object
 2   Review      9955 non-null   object
 3   Rating      9962 non-null   object
 4   Metadata    9962 non-null   object
 5   Time        9962 non-null   object
 6   Pictures    10000 non-null  int64 
dtypes: int64(1), object(6)
memory usage: 547.0+ KB


,0
Restaurant,0
Reviewer,38
Review,45
Rating,38
Metadata,38
Time,38
Pictures,0


**Data Cleaning**

In [4]:
# Remove rows where Review is missing
df_reviews = reviews.dropna(subset=['Review'])

# Remove rows where Rating is missing
df_reviews = df_reviews.dropna(subset=['Rating'])

# Convert Review text to lowercase
df_reviews['Review'] = df_reviews['Review'].str.lower()

# Check cleaned data
df_reviews.head()

,Restaurant,Reviewer,Review,Rating,Metadata,Time,Pictures
0,Beyond Flavours,Rusha Chakraborty,"the ambience was good, food was quite good . h...",5,"1 Review , 2 Followers",5/25/2019 15:54,0
1,Beyond Flavours,Anusha Tirumalaneedi,ambience is too good for a pleasant evening. s...,5,"3 Reviews , 2 Followers",5/25/2019 14:20,0
2,Beyond Flavours,Ashok Shekhawat,a must try.. great food great ambience. thnx f...,5,"2 Reviews , 3 Followers",5/24/2019 22:54,0
3,Beyond Flavours,Swapnil Sarkar,soumen das and arun was a great guy. only beca...,5,"1 Review , 1 Follower",5/24/2019 22:11,0
4,Beyond Flavours,Dileep,food is good.we ordered kodi drumsticks and ba...,5,"3 Reviews , 2 Followers",5/24/2019 21:37,0


**Merge Both Datasets**

In [5]:
# Merge reviews with metadata
df = pd.merge(df_reviews, meta, left_on='Restaurant', right_on='Name')

# Display merged data
df.head()

,Restaurant,Reviewer,Review,Rating,Metadata,Time,Pictures,Name,Links,Cost,Collections,Cuisines,Timings
0,Beyond Flavours,Rusha Chakraborty,"the ambience was good, food was quite good . h...",5,"1 Review , 2 Followers",5/25/2019 15:54,0,Beyond Flavours,https://www.zomato.com/hyderabad/beyond-flavou...,800,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...","12noon to 3:30pm, 6:30pm to 11:30pm (Mon-Sun)"
1,Beyond Flavours,Anusha Tirumalaneedi,ambience is too good for a pleasant evening. s...,5,"3 Reviews , 2 Followers",5/25/2019 14:20,0,Beyond Flavours,https://www.zomato.com/hyderabad/beyond-flavou...,800,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...","12noon to 3:30pm, 6:30pm to 11:30pm (Mon-Sun)"
2,Beyond Flavours,Ashok Shekhawat,a must try.. great food great ambience. thnx f...,5,"2 Reviews , 3 Followers",5/24/2019 22:54,0,Beyond Flavours,https://www.zomato.com/hyderabad/beyond-flavou...,800,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...","12noon to 3:30pm, 6:30pm to 11:30pm (Mon-Sun)"
3,Beyond Flavours,Swapnil Sarkar,soumen das and arun was a great guy. only beca...,5,"1 Review , 1 Follower",5/24/2019 22:11,0,Beyond Flavours,https://www.zomato.com/hyderabad/beyond-flavou...,800,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...","12noon to 3:30pm, 6:30pm to 11:30pm (Mon-Sun)"
4,Beyond Flavours,Dileep,food is good.we ordered kodi drumsticks and ba...,5,"3 Reviews , 2 Followers",5/24/2019 21:37,0,Beyond Flavours,https://www.zomato.com/hyderabad/beyond-flavou...,800,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...","12noon to 3:30pm, 6:30pm to 11:30pm (Mon-Sun)"


We join both datasets using Restaurant (from reviews) and Name (from metadata)

**Text Preprocessing (NLP)**

In [6]:
# Download stopwords
nltk.download('stopwords')

# Load stopwords
stop_words = set(stopwords.words('english'))

# Function to clean text
def clean_text(text):
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Split into words
    words = text.split()

    # Remove stopwords
    words = [word for word in words if word not in stop_words]

    # Join words back into sentence
    return ' '.join(words)

# Apply cleaning function
df['cleaned_review'] = df['Review'].apply(clean_text)

# Check result
df[['Review', 'cleaned_review']].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,Review,cleaned_review
0,"the ambience was good, food was quite good . h...",ambience good food quite good saturday lunch c...
1,ambience is too good for a pleasant evening. s...,ambience good pleasant evening service prompt ...
2,a must try.. great food great ambience. thnx f...,must try great food great ambience thnx servic...
3,soumen das and arun was a great guy. only beca...,soumen das arun great guy behavior sincerety g...
4,food is good.we ordered kodi drumsticks and ba...,food good ordered kodi drumsticks basket mutto...


**Download Stopwords**: Downloads a list of common words like the, is, and, in, on.
**Load Stopwords**: Stores stopwords in a set for faster checking.
**Create Cleaning Function**: This function cleans each review step-by-step.


**Create Target Variable**

In [7]:
# Convert Rating column to numeric
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

# Check again
df['Rating'].head()

,Rating
0,5.0
1,5.0
2,5.0
3,5.0
4,5.0


In [8]:
df['Rating'].value_counts()

,count
Rating,
5.0,3826
4.0,2373
1.0,1735
3.0,1192
2.0,684
4.5,69
3.5,47
2.5,19
1.5,9


In [9]:
df[['Rating']].sample(10)

,Rating
5918,5.0
1260,5.0
4443,3.0
6467,5.0
3622,5.0
3950,4.0
8146,4.0
4938,1.0
1841,4.0
7014,1.0


**Train-Test Split**

In [14]:
# Create the 'sentiment' column based on 'Rating'
df['sentiment'] = df['Rating'].apply(lambda rating: 1 if rating >= 3 else 0)

# Define input (X) and output (y)
X = df['cleaned_review']
y = df['sentiment']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Check sizes
print("Training data size:", X_train.shape)
print("Testing data size:", X_test.shape)

Training data size: (7964,)
Testing data size: (1991,)


X (Input Features)

Contains cleaned review text
This is what the model will learn from

y (Target Variable)

Contains sentiment (0 or 1)
This is what the model will predict

We split the data into training and testing sets to evaluate the model’s performance on unseen data.

**Convert Text to Numbers (TF-IDF)**

In [15]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000)

# Fit and transform training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform testing data
X_test_tfidf = tfidf.transform(X_test)

Term Frequency – Inverse Document Frequency: Gives importance to words based on:
How often they appear in a review.
How unique they are across all reviews.

 TF-IDF helps in identifying important words in reviews by assigning higher weights to meaningful terms and lower weights to common words.

**Train Machine Learning Model**

In [16]:
# Initialize model
model = RandomForestClassifier()

# Train the model
model.fit(X_train_tfidf, y_train)

RandomForestClassifier()

We are using Random Forest Classifier because it is a collection of many decision trees each tree gives a prediction final output = majority vote.


Model sees:

Input → TF-IDF features (numbers)

Output → Sentiment (0 or 1)
Example:
"amazing", "good" → Positive

"bad", "slow" → Negative


**Prediction & Model Evaluation**

In [17]:
# Make predictions
y_pred = model.predict(X_test_tfidf)

# Check accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# Detailed report
print(classification_report(y_test, y_pred))

Accuracy: 0.8895027624309392
              precision    recall  f1-score   support

           0       0.85      0.69      0.76       509
           1       0.90      0.96      0.93      1482

    accuracy                           0.89      1991
   macro avg       0.87      0.82      0.84      1991
weighted avg       0.89      0.89      0.89      1991



We evaluated the model using accuracy and classification metrics to ensure it performs well on unseen data.

The model uses test data (unseen data) to predict sentiment.
Measures how many predictions are correct.

Precision → “When model says positive, how often is it correct?”

Recall → “How many actual positives did model find?”

F1-score → Overall balance


**Test with Custom Input**

In [18]:
# Function to predict sentiment for new review
def predict_review(text):
    # Clean the text
    text = clean_text(text)

    # Convert to TF-IDF
    vector = tfidf.transform([text])

    # Predict
    prediction = model.predict(vector)

    # Output result
    if prediction[0] == 1:
        return "Positive 😊"
    else:
        return "Negative 😡"

# Test example
predict_review("Food was amazing but service was slow")

'Positive 😊'

This step allows the model to take new customer reviews as input and predict whether the sentiment is positive or negative. The input text is first cleaned and converted into numerical form using TF-IDF, and then the trained model generates the prediction. This makes the system interactive and useful for real-world applications.

**Different test reviews**

In [ ]:
print(predict_review("The food was absolutely delicious and service was great"))
print(predict_review("Worst experience ever, very slow service and bad food"))
print(predict_review("Average taste, nothing special"))
print(predict_review("Loved the ambience and tasty dishes"))
print(predict_review("Food was cold and staff was rude"))

**Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression()
model_lr.fit(X_train_tfidf, y_train)

y_pred_lr = model_lr.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

In [ ]:
plt.figure()

plt.pie(accuracies, labels=models, autopct='%1.2f%%')

plt.title("Model Accuracy Comparison (Pie Chart)")

plt.show()

In [ ]:
df['Name'].unique()[:20]

In [ ]:
def recommend_similar_restaurants(restaurant_name, top_n=5):

    # Find matching restaurant (partial match)
    matches = df[df['Name'].str.contains(restaurant_name, case=False, na=False)]

    if matches.empty:
        return "Restaurant not found in dataset"

    idx = matches.index[0]

    scores = list(enumerate(similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    top_indices = [i[0] for i in scores[1:top_n+1]]

    return df.iloc[top_indices][['Name', 'Rating']]

In [ ]:
recommend_similar_restaurants("domino")

This project successfully applies machine learning and NLP techniques to analyze restaurant reviews and predict customer sentiment. It also extends to an advanced recommendation system that suggests similar restaurants using TF-IDF and cosine similarity. The models were evaluated and compared to ensure reliable performance, making the system effective for real-world applications in understanding customer feedback and providing personalized recommendations.